# Reproduce a Gradavia dataset

Download one public snapshot, retain its source version and verify missing-value semantics. Python 3 standard library only. Run the cells in order. No database credentials or API key are required.

## Parameters

Set `FAMILY` to `parcoursup`, `apprentissage` or `apb`. Leave `CAMPAIGN` and `VERSION` empty to discover the latest snapshot, or paste a recorded release ID to reproduce it. These source families describe different populations and must remain separate.


In [ ]:
import json
import os
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

BASE_URL = os.environ.get("GRADAVIA_API_BASE_URL", "https://gradavia.com").rstrip("/")
FAMILY = "parcoursup"
CAMPAIGN = ""
VERSION = ""
OUTPUT_DIRECTORY = Path("gradavia-export")


## Download one snapshot

A request returns at most 30,000 source rows. An unavailable campaign, release or service fails explicitly rather than silently replacing the requested data.


In [ ]:
parameters = {"famille": FAMILY, "format": "json"}
if CAMPAIGN:
    parameters["campagne"] = CAMPAIGN
if VERSION:
    parameters["version"] = VERSION
url = f"{BASE_URL}/api/v1/datasets?{urlencode(parameters)}"
request = Request(url, headers={"Accept": "application/json"})
with urlopen(request, timeout=30) as response:
    payload = json.load(response)
if payload.get("status") != "ready":
    raise RuntimeError("No published dataset is available for these parameters")
data = payload["data"]
source = data["source"]
assert data["family"] == FAMILY
if VERSION:
    assert source["releaseId"].lower() == VERSION.lower()
if CAMPAIGN:
    assert source["campaign"] == int(CAMPAIGN)
parameters.update(campagne=str(source["campaign"]), version=source["releaseId"])
pinned_url = f"{BASE_URL}/api/v1/datasets?{urlencode(parameters)}"
print("Records:", len(data["items"]))
print("Campaign:", source["campaign"])
print("Reproducible URL:", pinned_url)
print("License:", source["license"])


## Validate identity and missing values

A source row is identified by its release and row number. This does not establish continuity across campaigns. Zero is observed; null is accompanied by `missing`, `suppressed` or `invalid`.


In [ ]:
rows = data["items"]
assert len({row["id"] for row in rows}) == len(rows)
assert all(row["id"].startswith(source["releaseId"] + ":") for row in rows)
for row in rows:
    for key, value in row["metrics"].items():
        state = row["states"].get(key, "observed")
        assert (value is not None) == (state == "observed")
        assert state in {"observed", "missing", "suppressed", "invalid"}
for coverage in data["coverage"]:
    assert sum(coverage[state] for state in ("observed", "missing", "suppressed", "invalid")) == len(rows)
print("Identity and missing-value checks passed.")


## Calculate an explicitly covered sum

Published capacity can be summed over source formation rows. Excluded values are not estimated. Summed candidatures do not represent unique people; do not average access rates or combine percentages with different denominators.


In [ ]:
capacity = [row["metrics"]["capacity"] for row in rows if row["metrics"]["capacity"] is not None]
summary = {"campaign": source["campaign"], "published_capacity": sum(capacity) if capacity else None, "observed_rows": len(capacity), "total_rows": len(rows)}
print(json.dumps(summary, ensure_ascii=False, indent=2))
for definition in data["definitions"]:
    if definition["key"] == "capacity":
        print(definition["field"], definition["description"])


## Keep the evidence

Save the downloaded snapshot and its version-pinned URL. Subsequent analyses can read this JSON without fetching newer data. Keep definitions, coverage, license and notices with any exported results.


In [ ]:
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
filename = OUTPUT_DIRECTORY / "{}-{}-{}.json".format(FAMILY, source["campaign"], source["releaseId"])
filename.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_DIRECTORY / "source-url.txt").write_text(pinned_url + "\n", encoding="utf-8")
print("Saved:", filename)
